# Your First Agent Test

You have a documentation-search agent in `documentation-agent/`. This lesson writes a **test** for it: a small program that runs the agent and checks, automatically, that the answer has the shape you expect. Run top to bottom. The lab comes first; the **Appendix** has tiny runnable examples for the new ideas. Open one only when a line confuses you.

```text
test_agent_runs()                              the test
      ↓
create_test_agent()                            builds tools + agent, like main.py
      ↓
await run_agent_stream(agent, "llm as a judge")   the model searches, reads files, answers  (slow, async)
      ↓
result.output                                  a RAGResponse object
      ↓
4 asserts: answer, confidence, found_answer, followup_questions
      ↓
pytest prints PASSED or FAILED
```

Think of a taste test before a dish goes on the menu: the `assert` lines are the checklist and **pytest** is the taster.

**New terms:** a **test** is a function named `test_*` that fails if something is wrong. `assert` means "this must be true". **pytest** finds and runs tests; **pytest-asyncio** lets tests be `async def`. A **dev dependency** (`uv add --dev`) is needed to develop and test but not to run the finished app.

**Already covered, so not repeated:** dataclasses and decorators, factories, the event loop in general (the Appendix has a short refresher on how it applies to tests).

**Differences from the video (all tested):**

| Video | Here | Why |
|---|---|---|
| Real-model test only | A free fake-model run first | Checks the wiring and the four assertions with no cost; it proves wiring, not answer quality |
| Run from the project folder | The notebook moves into the project first | The notebook lives in `tests/`; without moving, `.cache` would be created in `tests/` and the docs downloaded again |
| `.vscode/settings.json` makes tests appear in VS Code | Same, but VS Code reads it only if `documentation-agent/` is the opened workspace folder | Your workspace root is the parent folder |
| Edit `doc_agent.py` by hand | The notebook only **checks** the edit | Never silently change your files |

## 1. Setup

Finds the project, moves into it, and stops with a clear message if the kernel is wrong. **Select the kernel that uses `documentation-agent/.venv`.** Installing the test tools (already done in your project) is `uv add --dev pytest pytest-asyncio` from `documentation-agent/`.

In [ ]:
import importlib.util
import os
import sys
from pathlib import Path

# Show which Python runs this notebook.
print("Notebook Python:", sys.executable)


# Find the project folder (the one holding doc_agent.py and tools.py) from here upward.
def find_project_dir():
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "doc_agent.py").exists() and (folder / "tools.py").exists():
            return folder
        child = folder / "documentation-agent"
        if (child / "doc_agent.py").exists() and (child / "tools.py").exists():
            return child
    raise FileNotFoundError("Could not find doc_agent.py and tools.py. Open this notebook inside documentation-agent/tests/.")


PROJECT_DIR = find_project_dir()
# Work from the project folder and make "import tools" / "import doc_agent" work.
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print("Project directory:", PROJECT_DIR)

# Stop with one clear message if this kernel lacks a package the cells import.
# (pytest_asyncio is only needed when you run `uv run pytest`, so it is not required here.)
missing = [p for p in ("pydantic_ai", "pytest", "minsearch", "gitsource", "jaxn", "dotenv") if importlib.util.find_spec(p) is None]
if missing:
    raise RuntimeError(f"This kernel cannot import {missing}. Select {PROJECT_DIR / '.venv' / 'bin' / 'python'} as the kernel.")

# doc_agent.py calls AgentStream.stream_responses(), which exists in pydantic-ai 1.x but was renamed in 2.x.
from pydantic_ai.result import AgentStream
if not hasattr(AgentStream, "stream_responses"):
    raise RuntimeError(f"This kernel's pydantic-ai is too new for doc_agent.py. Select {PROJECT_DIR / '.venv' / 'bin' / 'python'} as the kernel.")

# Load OPENAI_API_KEY from a .env file if there is one (only the paid test needs it; real env values win).
import dotenv
dotenv.load_dotenv(PROJECT_DIR / ".env", override=False)
print("Environment OK | OPENAI_API_KEY is set:", bool(os.getenv("OPENAI_API_KEY")))

## 2. Confirm the return-type fix

The video's fix: in `run_agent_stream`, change `return agent_run` to `return agent_run.result`. `agent.iter(...)` gives a **run** object while the agent works; only the finished **result** has `.output` (Appendix A4). Your project already has the fix; this cell only **checks** it. `main.py` is unaffected, since it only uses `.new_messages()` and `.output`, which a result also has.

In [ ]:
# Read doc_agent.py as text (read only, never written).
source = (PROJECT_DIR / "doc_agent.py").read_text(encoding="utf-8")
has_fix = "return agent_run.result" in source
has_old = "return agent_run\n" in source
print("Has 'return agent_run.result':", has_fix, "| still has old 'return agent_run':", has_old)
# Stop clearly now, instead of a confusing AttributeError later.
assert has_fix and not has_old, "In run_agent_stream, change 'return agent_run' to 'return agent_run.result'."

## 3. The test, free: a fake model

`FunctionModel` replaces the language model with our own Python function. This one plays the model: it asks for a `search`, then returns the structured answer through the `final_result` tool. Two invented documents, so **no download, no key, no cost**. The fake run proves the wiring and the four assertions, not real answer quality.

Look for: `TOOL CALL (search): ...` is the model asking for a search (the `(search)` is the agent's name). The streamed `answer`, `answer type` and `follow up question` lines come from the project's real streaming code. `final_result` is how the model hands back the structured answer; it is not another documentation lookup.

In [ ]:
import json
from pydantic_ai.models.function import DeltaToolCall, FunctionModel
from doc_agent import DocumentationAgentConfig, create_agent, run_agent_stream, DEFAULT_INSTRUCTIONS
from tools import SearchTools, build_index, create_highlighter, create_documentation_tools_cached

# Two invented documents instead of a download.
fake_docs = [
    {"filename": "judge.md", "title": "LLM as a judge", "description": "Scoring outputs",
     "content": "Use an LLM as a judge to score model outputs against a rubric."},
    {"filename": "bare.md", "content": "A bare file without frontmatter."},
]
fake_tools = SearchTools(build_index(fake_docs), create_highlighter(), {d["filename"]: d["content"] for d in fake_docs})

# A structured answer with every RAGResponse field (models.py).
FAKE_ANSWER = {"answer": "Use an LLM as a judge to score outputs (judge.md).", "found_answer": True, "confidence": 0.9,
               "confidence_explanation": "Invented.", "answer_type": "how-to", "followup_questions": ["How do I write a rubric?"]}


# A fresh scripted model per run, so its turn counter starts at zero.
def make_fake_model():
    turn = {"n": 0}

    async def scripted(messages, info):
        turn["n"] += 1
        if turn["n"] == 1:
            yield {0: DeltaToolCall(name="search", json_args='{"query": "judge"}')}                  # turn 1: search
        else:
            yield {0: DeltaToolCall(name="final_result", json_args=json.dumps(FAKE_ANSWER))}        # turn 2: answer

    return FunctionModel(stream_function=scripted)


# The lesson's four checks, as a function that takes the agent so we can reuse it.
async def check_agent_response(agent, user_prompt):
    result = await run_agent_stream(agent, user_prompt)    # await: Jupyter already has an event loop
    search_result = result.output
    assert search_result.answer is not None                # an answer exists
    assert search_result.confidence >= 0.0                 # a confidence score exists
    assert search_result.found_answer is True              # the agent says it found something
    assert len(search_result.followup_questions) > 0       # follow-up suggestions exist
    return result


offline_agent = create_agent(DocumentationAgentConfig(model=make_fake_model(), instructions=DEFAULT_INSTRUCTIONS), fake_tools)
await check_agent_response(offline_agent, "llm as a judge")
print("\nOffline agent checks passed; no model calls made.")

## 4. The real test (paid, opt-in)

The same four checks with the real model, through `create_test_agent()`: a **factory** that builds the agent exactly as `main.py` does, so tests exercise the real wiring (it loads the saved index from `.cache`, or downloads the docs if missing). Real calls are billed, so this stays off until you turn it on.

```text
pytest starts a loop and awaits test_agent_runs
   ↓  create_test_agent()                  normal function (no await)
   ↓  await run_agent_stream(agent, ...)   async: the slow part
   ↓  result.output                        normal attribute: the RAGResponse
   ↓  4 asserts                            the first false one fails the test
```

In `tests/test_agent.py` the same body sits under `@pytest.mark.asyncio`, which gives the test an event loop to run on. The notebook doesn't need it. A pass shows the answer has the right *shape*, not that it is correct.

In [ ]:
# Build tools and an agent exactly like main.py does.
def create_test_agent():
    tools = create_documentation_tools_cached()
    return create_agent(DocumentationAgentConfig(instructions=DEFAULT_INSTRUCTIONS), tools)


async def test_agent_runs():
    result = await check_agent_response(create_test_agent(), "llm as a judge")
    print("\ntest_agent_runs passed")
    return result


# Real API calls cost money, so this stays off until you turn it on.
RUN_PAID_TEST = False

if RUN_PAID_TEST:
    assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY (in the environment or documentation-agent/.env) first."
    await test_agent_runs()
else:
    print("Skipped: set RUN_PAID_TEST = True to make real, billed model calls.")

## 5. Run it with pytest

From a terminal inside `documentation-agent/`:

```bash
uv run pytest
```

- `pythonpath = ["."]` in `pyproject.toml` lets pytest find `tools` and `doc_agent`. The `"."` means the folder holding `pyproject.toml` (pytest's rootdir), wherever you run it from (Appendix A5).
- `.vscode/settings.json` points VS Code's test panel at `tests` and turns on pytest. It only applies if `documentation-agent/` is the opened workspace folder.

The cell below only **lists** the tests (collect-only), so it is free.

In [ ]:
import subprocess

completed = subprocess.run([sys.executable, "-m", "pytest", "--collect-only", "-q", "-p", "no:cacheprovider"],
                           cwd=PROJECT_DIR, capture_output=True, text=True)
print(completed.stdout)
assert completed.returncode == 0, completed.stderr

## Recap, validation, production

- A test is a function of asserts; pytest finds `test_*` functions and runs them.
- Async code needs an event loop: `@pytest.mark.asyncio` in pytest, `await` in Jupyter.
- `run_agent_stream` must return `agent_run.result` so callers get `.output`.
- The fake-model test checks wiring for free; the real test checks the whole path and costs a few cents.

**Validated:** executed top to bottom in a scratch copy on `documentation-agent/.venv` (Python 3.13, pydantic-ai 1.59.0, pytest 9.1.1, pytest-asyncio 1.4.0) with `RUN_PAID_TEST = False`. Separately, `uv run pytest` passed against the real model from both `tests/` and `documentation-agent/`. **Not run:** this notebook's paid cell, the VS Code test panel, and Python 3.14.2.

### Short production reminder

For production, replace print-based callbacks with structured logs or tracing events. Include request/run IDs, tool name, latency, status, and redacted argument summaries. Do not log sensitive content or full tool arguments by default. Keep callbacks observational; enforce authorization and validation inside the tool execution boundary.

This test checks the answer's shape, not its correctness; `confidence` is a number the model wrote, not a measured accuracy. Before client use: keep paid tests behind a pytest marker, evaluate answers against reference questions, record cost and latency, and run tests in CI.

# Appendix: tiny examples for the new ideas

Standard library only (A2 also needs pytest in the kernel). No network, no cost. Skip them unless a line confused you.

### A1. `assert`

`assert condition` says "this must be true"; if it is false, Python raises `AssertionError`. A test is a function of asserts. The four checks in the agent test are asserts.

In [ ]:
answer = "Use an LLM as a judge"
confidence = 0.85
assert answer is not None        # passes silently
assert confidence >= 0.0         # passes silently
try:
    # The text after the comma is the failure message.
    assert len([]) > 0, "expected at least one follow-up question"
except AssertionError as error:
    print("Failed assert ->", error)

### A2. How pytest finds tests

pytest looks for `test_*.py` files and `test_*` functions, runs each, and reports passed or failed. It ignores other names. The `-s` flag shows `print` output while tests run (pytest hides it unless a test fails).

In [ ]:
import subprocess
import tempfile

DEMO_TESTS = ("def test_passes():\n    assert 1 + 1 == 2\n\n"
              "def test_fails():\n    assert 1 + 1 == 3, 'math is broken'\n\n"
              "def helper_not_a_test():\n    raise RuntimeError('never run')\n")
with tempfile.TemporaryDirectory() as folder:
    Path(folder, "test_demo.py").write_text(DEMO_TESTS, encoding="utf-8")
    done = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"], cwd=folder, capture_output=True, text=True)
print(done.stdout.strip().splitlines()[-1])   # expect: 1 failed, 1 passed

### A3. Coroutines, `await` and `@pytest.mark.asyncio`

Calling an `async def` function only creates a **coroutine** (paused work); `await` runs it. Jupyter already has an event loop, so you `await` at the top level. pytest has none, so `@pytest.mark.asyncio` (from pytest-asyncio) tells it to start a loop and await your test. Without that decorator the test body would never run.

In [ ]:
async def say_hello():
    return "hello"

coroutine = say_hello()                       # nothing has run yet
print("Type of the call result:", type(coroutine).__name__)
print("After await:", await coroutine)        # works in Jupyter because a loop is already running

### A4. The run vs its result

`run_agent_stream` used to return the **run** (no `.output`); the fix returns `agent_run.result` (has `.output`). Stand-in classes show it with no framework.

In [ ]:
class FakeResult:
    def __init__(self, output):
        self.output = output

class FakeRun:
    def __init__(self, result):
        self.result = result

run = FakeRun(FakeResult("final answer"))
try:
    print(run.output)
except AttributeError as error:
    print("Before the fix ->", error)
print("After the fix  ->", run.result.output)

### A5. `sys.path`, `pythonpath` and the rootdir

`import tools` searches the folders in `sys.path`. The tests live in `tests/`, but `tools.py` is one folder up, so `pythonpath = ["."]` adds the project folder. `"."` is relative to pytest's **rootdir** (the folder with `pyproject.toml`), not your terminal's folder. `PYTHONPATH=. uv run pytest` is different: there `.` is the folder you run it from.

In [ ]:
import importlib

with tempfile.TemporaryDirectory() as folder:
    Path(folder, "my_demo_module.py").write_text("GREETING = 'hello from the module'\n", encoding="utf-8")
    try:
        importlib.import_module("my_demo_module")
    except ModuleNotFoundError as error:
        print("Before adding the folder ->", error)
    sys.path.insert(0, folder)                 # what pythonpath = ["."] does for pytest
    print("After adding the folder  ->", importlib.import_module("my_demo_module").GREETING)
    sys.path.remove(folder)                    # clean up
    sys.modules.pop("my_demo_module", None)

### A6. Your note: the event loop runs two jobs at once

From your own study notes. `await` marks where a job can step aside; `asyncio.gather` asks the loop to run both jobs, switching whenever one is waiting. Two 2-second waits finish in about 2 seconds, not 4. Replace `asyncio.sleep` with `time.sleep` (a blocking wait) and it takes 4.

In [ ]:
import asyncio, time

async def order(name, seconds):
    print(f"{name}: started")
    await asyncio.sleep(seconds)     # "I'm waiting": the loop can run something else
    print(f"{name}: done")
    return name

start = time.perf_counter()
# gather asks the loop to run both jobs, switching whenever one is waiting
results = await asyncio.gather(order("pasta", 2), order("soup", 2))
print(results, f"took {time.perf_counter() - start:.1f}s")